# Aprendizado de Máquina — Aula prática 03

## Seleção de Modelos e Validação Cruzada

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

As duas primeiras aulas práticas terminaram devendo a mesma coisa. Na Aula 01
escolhemos o grau do polinômio olhando o erro no conjunto de teste; na Aula 02,
o $\lambda$ do Lasso do mesmo jeito. Nas duas vezes ficou dito que aquilo era
trapaça e que a resposta viria na Aula 03. Chegamos nela.

O problema é sempre o mesmo:

> **estimar o risco de um procedimento usando os dados que temos, sem nunca
> avaliar um modelo nos pontos que o treinaram.**

E de novo vamos trabalhar na população sintética da Aula 01, porque nela — e só
nela — dá para conferir a resposta: conhecemos $r(x)$ e $\sigma^2$, logo sabemos
o risco verdadeiro de cada modelo. A validação cruzada vai ser julgada contra ele.
No fim abrimos o `superconductivity.csv` e escolhemos um hiperparâmetro no escuro,
que é como a coisa acontece de verdade.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- mostrar, numericamente, que o erro de treino é um estimador otimista do risco;
- implementar *data splitting*, LOOCV e $k$-dobras **na mão**, e reencontrar com
  eles os números que o `scikit-learn` devolve;
- verificar o atalho da alavancagem que dá o LOOCV com um único ajuste;
- reconhecer a armadilha de rodar $k$-dobras sem embaralhar;
- julgar a curva de validação cruzada contra o risco verdadeiro, e desempatar
  modelos pela regra de um erro-padrão;
- executar o protocolo da aula teórica — separar o teste, escolher por CV
  **dentro do treino**, e tocar o teste uma única vez —, e medir o que se ganha
  e o que se paga com ele;
- reconhecer por que o mínimo da CV não pode ser usado para reportar desempenho.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos desta aula são os de `sklearn.model_selection` — é o módulo
inteiro dedicado a separar dados e estimar risco.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. A população da Aula 01, de volta

Mesma função de regressão, mesmo ruído, mesmo tamanho de amostra:

$$X \sim \mathrm{Unif}[-3,3], \qquad Y = \underbrace{\sin(1{,}5X) + 0{,}3X}_{r(X)} + \varepsilon,
  \qquad \varepsilon \sim N(0, 0{,}7^2).$$

O erro irredutível é $\sigma^2 = 0{,}49$: nenhum método, por melhor que seja,
consegue erro quadrático médio abaixo disso.

In [ ]:
def r(x):
    return np.sin(1.5 * x) + 0.3 * x


SIGMA = 0.7           # erro irredutivel = 0.49
A, B = -3.0, 3.0      # suporte de X
N_TR = 50             # tamanho da amostra


def amostra(n, rng):
    x = rng.uniform(A, B, size=n)
    y = r(x) + rng.normal(0, SIGMA, size=n)
    return x, y

A classe de modelos também é a mesma: polinômios de grau $p$, ajustados por
mínimos quadrados. O `StandardScaler` no meio do caminho não muda o ajuste — é
só condicionamento numérico, para que $x^{12}$ não estoure.

In [ ]:
def modelo_poly(grau):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=grau, include_bias=False)),
        ("escala", StandardScaler()),
        ("mqo", skl.LinearRegression()),
    ])


rng = np.random.default_rng(6)
x, y = amostra(N_TR, rng)
X = x.reshape(-1, 1)

grade = np.linspace(A, B, 400)
fig, ax = subplots(figsize=(5.2, 3.2))
ax.scatter(x, y, s=18, alpha=0.75, label="a amostra (n = 50)")
ax.plot(grade, r(grade), lw=2, color="crimson", label="r(x), que so nos conhecemos")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(fontsize=8)

---
## 3. Por que o erro de treino não serve

A pergunta que abre a aula: *se eu ajustar um polinômio de grau 50 a 50 pontos,
qual o erro no treino?* Vamos responder subindo o grau devagar e olhando os dois
números — o erro no treino e o erro numa amostra nova, grande, que serve de
teste.

In [ ]:
x_te, y_te = amostra(20_000, np.random.default_rng(99))
X_te = x_te.reshape(-1, 1)

graus = np.arange(1, 21)
tabela = []
for g in graus:
    m = modelo_poly(g).fit(X, y)
    tabela.append({
        "grau": g,
        "erro no treino": np.mean((y - m.predict(X)) ** 2),
        "erro no teste": np.mean((y_te - m.predict(X_te)) ** 2),
    })

tabela = pd.DataFrame(tabela).set_index("grau")
tabela.iloc[[0, 2, 4, 9, 14, 17, 18, 19]].round(4)

O erro de treino desce **monotonicamente** e o de teste, não. Um critério que dá
nota máxima ao pior modelo da lista não serve como critério — é esse o ponto
inteiro da aula.

Repare também que o erro de treino está abaixo de $\sigma^2 = 0{,}49$ para
praticamente todos os graus, inclusive os bons. Isso é impossível para o risco de
verdade, que tem $\sigma^2$ como piso. Não é sintoma de superajuste: é o
**otimismo**, e ele é sistemático e previsível. Ajustando $p+1$ parâmetros por
mínimos quadrados a $n$ pontos, gasta-se $p+1$ graus de liberdade, e o erro de
treino esperado encolhe para

$$\mathbb{E}[\mathrm{EQM}_{\text{treino}}] \approx \sigma^2\left(1 - \frac{p+1}{n}\right),$$

desde que o modelo já seja flexível o bastante para o viés ser desprezível.
Vamos conferir isso em 300 amostras.

In [ ]:
rng_treino = np.random.default_rng(31)
soma = np.zeros(len(graus))
for _ in range(300):
    xb, yb = amostra(N_TR, rng_treino)
    Xb = xb.reshape(-1, 1)
    for j, g in enumerate(graus):
        soma[j] += np.mean((yb - modelo_poly(g).fit(Xb, yb).predict(Xb)) ** 2)

comparacao = pd.DataFrame({
    "erro de treino medio": soma / 300,
    "sigma^2 (1 - (p+1)/n)": SIGMA ** 2 * (1 - (graus + 1) / N_TR),
}, index=graus)
comparacao.index.name = "grau"
comparacao.loc[[1, 2, 3, 5, 8, 10, 15, 20]].round(4)

Do grau 5 em diante as duas colunas ficam a menos de 1,5% uma da outra. Nos graus
1 a 3 o erro de treino é bem maior que a fórmula prevê, e por um motivo que a
Aula 01 já tinha nomeado: ali o modelo é rígido demais, o viés domina, e o erro
de treino está medindo falta de flexibilidade em vez de ruído.

A lição: o erro de treino não erra por acidente, erra por construção, e o
tamanho do erro é $\sigma^2(p+1)/n$ — cresce exatamente com a complexidade do
modelo. É por isso que ele não pode arbitrar entre modelos de complexidades
diferentes.

In [ ]:
fig, ax = subplots(figsize=(5.2, 3.2))
ax.plot(tabela.index, tabela["erro no treino"], "^--", ms=4, label="erro no treino")
ax.plot(tabela.index, tabela["erro no teste"], "o-", ms=4, label="erro no teste")
ax.axhline(SIGMA ** 2, ls="--", lw=1, color="green", label="sigma^2 = 0,49")
ax.set_xlabel("grau do polinomio"); ax.set_ylabel("erro quadratico medio")
ax.set_xticks(graus[::2]); ax.set_ylim(0, 2.0)
ax.legend(fontsize=8)

A pergunta que abriu a seção pedia um polinômio de grau 50 para 50 pontos. Vamos ao
grau 49 — o maior que cabe aqui sem a coluna constante — e olhar duas coisas: o erro
de treino, e o que a curva faz **entre** os pontos observados.

In [ ]:
m49 = modelo_poly(49).fit(X, y)

print(f"EQM de treino, grau 49: {np.mean((y - m49.predict(X)) ** 2):.4f}")
print(f"EQM de treino, grau  5: {np.mean((y - modelo_poly(5).fit(X, y).predict(X)) ** 2):.4f}")
print(f"EQM de teste,  grau 49: {np.mean((y_te - m49.predict(X_te)) ** 2):.3g}")
print(f"posto que o lstsq enxergou: {m49['mqo'].rank_} de 49 colunas")

grade_fina = np.linspace(A, B, 400)
pred49 = m49.predict(grade_fina.reshape(-1, 1))
print(f"predicao na grade: de {pred49.min():.3g} a {pred49.max():.3g}")
print(f"y observado      : de {y.min():.2f} a {y.max():.2f}")

fig, ax = subplots(figsize=(5.6, 3.4))
ax.scatter(x, y, s=18, alpha=0.75, zorder=3, label="a amostra (n = 50)")
ax.plot(grade_fina, r(grade_fina), lw=2, color="crimson", label="r(x)")
ax.plot(grade_fina, pred49, lw=1.4, color="darkorange", label="polinomio de grau 49")
ax.set_ylim(-6, 6)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("cortado em y = -6: a curva desce ate -1,05e8", fontsize=9)
ax.legend(fontsize=8)

**O erro de treino não é zero — é $0{,}1029$.** No papel deveria ser: 50 pontos e 49
covariáveis ($x, x^2, \dots, x^{49}$) dão um sistema com solução exata, e a curva
deveria passar por todas as observações. Não passa, e o motivo é numérico: $x^{49}$
com $x\in[-3,3]$ varre umas vinte e quatro ordens de grandeza, a matriz perde posto
em ponto flutuante — o `lstsq` por trás do `LinearRegression` enxergou 39 colunas
independentes, não 49 — e ele devolve a solução de **norma mínima** em vez da que
interpola.

O que a moral da seção precisa sobreviveu intacto: $0{,}1029$ ainda é menor que o
$0{,}3580$ do grau 5. O erro de treino continua premiando o modelo mais flexível,
mesmo quando ele é indefensável.

E ele é indefensável. A amostra vive entre $-2{,}37$ e $2{,}07$; entre dois pontos
vizinhos a curva mergulha a $-1{,}05\times 10^{8}$. Fora da amostra o EQM é
$2{,}5\times 10^{13}$ — mais de treze ordens de grandeza acima do erro irredutível
$\sigma^2=0{,}49$. O ajuste é excelente exatamente nos 50 lugares onde ninguém
precisa dele.

> **Se a sua máquina imprimir outros números, está tudo certo.**

---
## 4. *Data splitting*: a solução mais simples

Separe os dados em duas partes, ajuste numa e avalie na outra. É honesto: a parte
separada não participou do ajuste, então a média dos erros nela é um estimador
consistente do risco. (As notas chamam essa parte de **conjunto de teste**; aqui as
variáveis levam o sufixo `vl`, de *validation*, mas o papel é o mesmo — ficar de
fora.)

Só que tem dois defeitos, e os dois são mensuráveis. Vamos medi-los.

In [ ]:
x_tr, x_vl, y_tr, y_vl = skm.train_test_split(x, y, test_size=0.3, random_state=0)

m = modelo_poly(5).fit(x_tr.reshape(-1, 1), y_tr)
estimativa = np.mean((y_vl - m.predict(x_vl.reshape(-1, 1))) ** 2)
print(f"treino: {len(x_tr)} pontos   validacao: {len(x_vl)} pontos")
print(f"risco estimado (grau 5): {estimativa:.4f}")

**Defeito 1: instabilidade.** Esse número dependeu da divisão sorteada. Vamos
sortear 200 divisões diferentes e olhar a distribuição das estimativas.

In [ ]:
estimativas = []
for semente in range(200):
    xa, xb, ya, yb = skm.train_test_split(x, y, test_size=0.3, random_state=semente)
    mm = modelo_poly(5).fit(xa.reshape(-1, 1), ya)
    estimativas.append(np.mean((yb - mm.predict(xb.reshape(-1, 1))) ** 2))
estimativas = np.array(estimativas)

print(f"media  das 200 estimativas: {estimativas.mean():.4f}")
print(f"desvio das 200 estimativas: {estimativas.std(ddof=1):.4f}")
print(f"minimo: {estimativas.min():.4f}   maximo: {estimativas.max():.4f}")

fig, ax = subplots(figsize=(5.2, 2.8))
ax.hist(estimativas, bins=30, color="steelblue", alpha=0.85)
ax.set_xlabel("risco estimado por data splitting (grau 5)")
ax.set_ylabel("frequencia")

A mesma amostra, o mesmo modelo, e a estimativa passeia por uma faixa larga. Quem
reportasse o mínimo estaria mentindo por baixo; quem reportasse o máximo, por
cima. E nada nos dados diz qual das 200 divisões é a "certa" — todas são
igualmente legítimas.

**Defeito 2: desperdício.** O modelo avaliado foi treinado com 35 pontos, não com
50. Não é o modelo que você vai usar no final. Com poucos dados isso importa, e
importa na direção pessimista.

A validação cruzada resolve os dois usando **todo o conjunto de treinamento** — e
vale ler isso com cuidado, porque não é "usando a amostra inteira". A divisão
treino/teste continua sendo o primeiro passo, e o teste continua guardado. O que a
CV dispensa é um *segundo* corte, o que separaria um pedaço do treino para escolher
o modelo: esse papel passa a ser das dobras, girando dentro do treino.

As Seções 5 a 7 vão rodar a CV nos 50 pontos assim mesmo, e por um motivo que não é
esse: lá o objeto de estudo é a **régua** — queremos comparar a CV com o atalho da
alavancagem, com o laço feito à mão e com o risco verdadeiro, e a resposta vem da
simulação. Quando voltarmos a agir como analista, na Seção 8, a divisão volta.

---
## 5. LOOCV e o atalho que sai de graça

Deixe **uma** observação de fora por vez:

$$\widehat R_{\text{LOO}} = \frac1n \sum_{i=1}^n \big(Y_i - g_{-i}(X_i)\big)^2 .$$

Custa $n$ ajustes. Com $n=50$ isso é trivial; com $n=10^5$, não. Mas para
qualquer ajuste **linear** existe uma fórmula fechada que devolve o LOOCV exato a
partir de um **único** ajuste, usando a diagonal da matriz de projeção
$\bm H = \bm Z(\bm Z^\top \bm Z)^{-1}\bm Z^\top$:

$$\widehat R_{\text{LOO}}
  = \frac1n \sum_{i=1}^n \left(\frac{Y_i - \widehat g(X_i)}{1 - h_{ii}}\right)^2 .$$

Não vamos aceitar isso de graça — vamos conferir.

In [ ]:
GRAU = 5
Z = PolynomialFeatures(degree=GRAU).fit_transform(X)   # inclui a coluna de 1s
H = Z @ np.linalg.pinv(Z)
h = np.diag(H)
y_ajustado = H @ y

atalho = np.mean(((y - y_ajustado) / (1 - h)) ** 2)

forca_bruta = 0.0
for i in range(N_TR):
    fora = np.arange(N_TR) != i
    beta = np.linalg.lstsq(Z[fora], y[fora], rcond=None)[0]
    forca_bruta += (y[i] - Z[i] @ beta) ** 2
forca_bruta /= N_TR

print(f"LOOCV por forca bruta (50 ajustes): {forca_bruta:.10f}")
print(f"LOOCV pelo atalho     (1 ajuste)  : {atalho:.10f}")
print(f"diferenca: {abs(atalho - forca_bruta):.3e}")

Coincidem até a precisão da máquina. Vale ler o que a fórmula está dizendo: o
resíduo de validação é o resíduo de treino **inflado** por $1/(1-h_{ii})$. E
$h_{ii}$ — a *alavancagem* — mede o quanto a observação $i$ puxa a própria
predição. Quanto mais alavancada a observação, mais o resíduo de treino a
subestima.

Dá para ver isso: a alavancagem é maior nas pontas do intervalo, onde há menos
vizinhos para segurar a curva.

In [ ]:
fig, ax = subplots(figsize=(5.2, 2.9))
ax.scatter(x, h, s=20, color="darkorange")
ax.axhline((GRAU + 1) / N_TR, ls="--", lw=1, color="gray",
           label="media = (p+1)/n")
ax.set_xlabel("x"); ax.set_ylabel("alavancagem  h_ii")
ax.legend(fontsize=8)

mais_alavancados = np.argsort(h)[-3:]
print("os 3 pontos mais alavancados estao em x =", np.round(x[mais_alavancados], 2))
print(f"neles, o fator de inflacao 1/(1-h) vale "
      f"{np.round(1 / (1 - h[mais_alavancados]), 2)}")

---
## 6. $k$ dobras, na mão e no `scikit-learn`

Embaralhe, corte em $k$ lotes, e use cada lote uma vez como validação. Cada
observação é validada exatamente uma vez e treina $k-1$ vezes. Primeiro na mão,
para não haver mistério:

In [ ]:
dobras = skm.KFold(5, shuffle=True, random_state=0)

erros_por_dobra = []
for treino, validacao in dobras.split(X):
    mm = modelo_poly(GRAU).fit(X[treino], y[treino])
    erro = np.mean((y[validacao] - mm.predict(X[validacao])) ** 2)
    erros_por_dobra.append(erro)
    print(f"treino: {len(treino):2d} pts   validacao: {len(validacao):2d} pts"
          f"   EQM = {erro:.4f}")

na_mao = np.mean(erros_por_dobra)
print(f"\nCV de 5 dobras, na mao: {na_mao:.4f}")

Agora a mesma coisa com uma linha. Duas coisas para reparar no resultado: o
`scikit-learn` segue a convenção *"maior é melhor"*, então o EQM aparece
**negado** — daí o sinal de menos; e o objeto `dobras` é o mesmo, com a mesma
semente, então o corte é idêntico e os números têm de bater exatamente.

In [ ]:
pontuacoes = skm.cross_val_score(modelo_poly(GRAU), X, y, cv=dobras,
                                 scoring="neg_mean_squared_error")
print("EQM por dobra:", np.round(-pontuacoes, 4))
print(f"do sklearn: {-pontuacoes.mean():.4f}   na mao: {na_mao:.4f}")

Um detalhe que quase nunca aparece. O laço que você escreveu guardou um
$\mathrm{EQM}_j$ por dobra e tirou a média dos cinco — e é exatamente isso que o
`cross_val_score` faz, e é a fórmula 5.3 do [ISLP]. Já a Equação 4 da nota, que é a
do [AME], soma os $n$ erros e divide por $n$. Agrupando por dobra, ela vira

$$\widehat R_{k\text{-CV}} = \sum_{j=1}^{k} \frac{|L_j|}{n}\,\mathrm{EQM}_j,$$

a mesma média dos mesmos $\mathrm{EQM}_j$, só que **ponderada pelo tamanho das
dobras**. Quando $k$ divide $n$ todos os pesos valem $1/k$ e as duas coincidem
exatamente — é o caso aqui ($50 = 5 \times 10$), e é por isso que os números
bateram.

Quando $k$ não divide $n$, elas diferem. Vale medir quanto.

In [ ]:
def duas_contas(X, y, grau, cv):
    """(a) soma os n erros e divide por n;  (b) media das medias das dobras."""
    soma, mses = 0.0, []
    for tr, te in cv.split(X):
        e2 = (y[te] - modelo_poly(grau).fit(X[tr], y[tr]).predict(X[te])) ** 2
        soma += e2.sum()
        mses.append(e2.mean())
    return soma / len(y), float(np.mean(mses))


cv7 = skm.KFold(7, shuffle=True, random_state=0)      # 50 nao e multiplo de 7
agrupada, media_dobras = duas_contas(X, y, GRAU, cv7)

print("tamanhos das dobras:", [len(te) for _, te in cv7.split(X)])
print(f"(a) somando os n erros e dividindo por n: {agrupada:.6f}")
print(f"(b) media das medias das dobras         : {media_dobras:.6f}")
print(f"    o cross_val_score faz a (b)         : "
      f"{-skm.cross_val_score(modelo_poly(GRAU), X, y, cv=cv7, scoring='neg_mean_squared_error').mean():.6f}")
print(f"    diferenca relativa                  : "
      f"{abs(agrupada - media_dobras) / media_dobras:.2%}")

In [ ]:
graus_d = np.arange(1, 11)
rng_d = np.random.default_rng(11)

discordam = 0
for b in range(300):
    xb, yb = amostra(N_TR, rng_d)
    Xb = xb.reshape(-1, 1)
    cv_b = skm.KFold(7, shuffle=True, random_state=b)
    par = np.array([duas_contas(Xb, yb, g, cv_b) for g in graus_d])
    if graus_d[np.argmin(par[:, 0])] != graus_d[np.argmin(par[:, 1])]:
        discordam += 1

print(f"em 300 amostras, escolhendo o grau por uma e por outra formula,")
print(f"elas discordam em {discordam} delas")

Com $k=7$ as dobras saem com 8 e 7 observações — desequilíbrio de 14% entre a maior
e a menor — e as duas contas divergem em $0{,}9\%$. O desequilíbrio nunca passa de
**uma** observação, então encolhe como $k/n$: nos 21.263 pontos do
`superconductivity.csv` ele seria de 0,03%, e a divergência entre as fórmulas,
invisível.

Mas *quase sempre desprezível* não é *sempre*. Escolhendo o grau do polinômio por
uma e por outra, em 300 amostras, elas discordam em **9**. Não é que uma esteja
certa e a outra errada — as duas são estimativas legítimas do mesmo risco. É que
perto do mínimo a curva de CV é plana, e 1% de diferença já basta para trocar o
argmin: o mesmo fenômeno da Seção 7, visto por outro ângulo.

A regra prática que sai daqui é barata: **com $n$ pequeno, escolha um $k$ que divida
$n$** — e a pergunta não se coloca.

### A armadilha do `shuffle`

`KFold(5)` sem `shuffle=True` corta os dados **na ordem em que estão no arquivo**.
Se o banco veio ordenado por alguma variável — e quem montou o banco pode ter
ordenado por um motivo que você não conhece —, cada dobra vira um pedaço
sistemático do domínio, e o modelo é obrigado a **extrapolar** para prever nela.

Vamos simular exatamente isso: ordenar a amostra por $x$ e rodar as duas versões.

In [ ]:
ordem = np.argsort(x)
X_ord, y_ord = X[ordem], y[ordem]

sem = -skm.cross_val_score(modelo_poly(GRAU), X_ord, y_ord, cv=skm.KFold(5),
                           scoring="neg_mean_squared_error")
com = -skm.cross_val_score(modelo_poly(GRAU), X_ord, y_ord,
                           cv=skm.KFold(5, shuffle=True, random_state=0),
                           scoring="neg_mean_squared_error")

print("banco ordenado por x, KFold(5) SEM shuffle:")
print("   EQM por dobra:", np.round(sem, 2))
print(f"   media: {sem.mean():.3f}")
print("\nos MESMOS dados, KFold(5) COM shuffle:")
print("   EQM por dobra:", np.round(com, 3))
print(f"   media: {com.mean():.3f}")

Os dados são os mesmos, o modelo é o mesmo, e as duas estimativas do risco não
estão nem na mesma ordem de grandeza. Sem embaralhar, o procedimento avaliado
deixou de ser *"prever $Y$ a partir de $X$"* e virou *"extrapolar para uma região
de $x$ que o treino nunca viu"* — um problema diferente, e muito mais difícil.

Em `train_test_split` o `shuffle=True` já é o padrão; em `KFold`, **não é**.
Construa o `KFold` explicitamente e ligue o `shuffle`.

---
## 7. A validação cruzada funciona?

Agora a pergunta que só a simulação responde: a curva de CV, calculada a partir
de **uma** amostra de 50 pontos, encontra o mesmo mínimo que o risco verdadeiro?

Para saber o risco verdadeiro precisamos de muitas amostras. E há um truque que
vale a pena conhecer: não é preciso sortear ruído de teste. Como

$$\mathbb{E}\big[(Y - \widehat r(x))^2 \mid X = x\big] = \big(r(x) - \widehat r(x)\big)^2 + \sigma^2,$$

basta comparar a predição com $r$ e somar $\sigma^2$ no fim. Isso elimina uma
fonte inteira de variabilidade de Monte Carlo, e as curvas saem limpas com muito
menos repetições.

In [ ]:
def risco_verdadeiro(graus, n_rep=300, semente=5, n_grade=400):
    """Risco de cada grau, por Monte Carlo sobre amostras de treino."""
    rng = np.random.default_rng(semente)
    x0 = np.linspace(A, B, n_grade)
    r0 = r(x0)
    X0 = x0.reshape(-1, 1)

    somas = np.zeros(len(graus))
    for _ in range(n_rep):
        xb, yb = amostra(N_TR, rng)
        Xb = xb.reshape(-1, 1)
        for j, g in enumerate(graus):
            pred = modelo_poly(g).fit(Xb, yb).predict(X0)
            somas[j] += np.mean((pred - r0) ** 2)
    return somas / n_rep + SIGMA ** 2


graus = np.arange(1, 11)
verdade = risco_verdadeiro(graus)
print("risco verdadeiro por grau:", np.round(verdade, 3))

E agora o que o analista realmente tem: uma amostra, e mais nada.

In [ ]:
cv_media, cv_ep, treino = [], [], []
for g in graus:
    p = -skm.cross_val_score(modelo_poly(g), X, y, cv=dobras,
                             scoring="neg_mean_squared_error")
    cv_media.append(p.mean())
    cv_ep.append(p.std(ddof=1) / np.sqrt(len(p)))
    treino.append(np.mean((y - modelo_poly(g).fit(X, y).predict(X)) ** 2))

cv_media, cv_ep = np.array(cv_media), np.array(cv_ep)

g_cv = graus[np.argmin(cv_media)]
g_verdade = graus[np.argmin(verdade)]
print(f"minimo do risco verdadeiro : grau {g_verdade}")
print(f"minimo da validacao cruzada: grau {g_cv}")

In [ ]:
fig, ax = subplots(figsize=(5.6, 3.4))
ax.plot(graus, verdade, "o-", ms=4, color="crimson",
        label="risco verdadeiro (inacessivel)")
ax.errorbar(graus, cv_media, yerr=cv_ep, fmt="s-", ms=4, capsize=2.5,
            color="steelblue", label="validacao cruzada, 5 dobras")
ax.plot(graus, treino, "^--", ms=4, color="gray", label="erro de treino")
ax.axhline(SIGMA ** 2, ls="--", lw=1, color="green", label="sigma^2")
ax.axvline(g_cv, ls=":", lw=1, color="steelblue")
ax.set_xlabel("grau do polinomio"); ax.set_ylabel("erro quadratico medio")
ax.set_xticks(graus); ax.set_ylim(0.15, 1.65)
ax.legend(fontsize=7.5)

Três leituras dessa figura, em ordem de importância.

1. **A CV acerta o lugar.** Ela põe o mínimo no grau 4, o risco verdadeiro põe no
   grau 5. Um grau de diferença, com uma amostra de 50 pontos.
2. **A CV erra o nível, e erra para os dois lados.** Nos graus baixos ela fica
   abaixo da curva vinho; nos graus 8 e 9, bem acima. E isso *não é um problema*
   para seleção de modelos: o que importa é **onde** está o mínimo, não quanto
   vale. É exatamente a observação que o [ISLP] faz na Figura 5.6, em que as
   curvas de CV ora subestimam, ora superestimam o risco, e todas identificam
   corretamente o nível de flexibilidade certo.
3. **A CV avisa quando está insegura.** As barras de um erro-padrão se abrem à
   direita, exatamente onde os modelos ficam instáveis.

E falta uma coisa. Escolhemos o grau 4 usando os 50 pontos, e o EQM daquele mínimo
é a única estimativa de risco que temos — só que ela é a estimativa que *escolheu*,
e por isso não pode ser a que *reporta*. Enquanto o assunto era estudar a régua
contra uma verdade conhecida, rodar a CV na amostra inteira estava certo; para
entregar um modelo e um número, não está. A Seção 8 troca de papel e roda o
protocolo da aula teórica.

### A regra de um erro-padrão

Como a estimativa tem erro-padrão, os graus 3 a 6 estão empatados dentro do
ruído. A **regra de 1-EP** resolve o empate pelo lado da parcimônia: escolha o
modelo mais simples cujo risco estimado esteja a no máximo um erro-padrão do
mínimo.

In [ ]:
limite = cv_media.min() + cv_ep[np.argmin(cv_media)]
g_1ep = graus[np.argmax(cv_media <= limite)]
print(f"minimo da CV: grau {g_cv}  (EQM {cv_media.min():.4f})")
print(f"limite de 1 erro-padrao: {limite:.4f}")
print(f"escolha pela regra de 1-EP: grau {g_1ep}")

Entre o grau 4 e o grau 5, na figura acima, o EQM de validação cruzada difere em
$0{,}003$. Vale perguntar quanto disso é sinal. Refazemos a curva inteira em outras
três amostras, mudando só a semente da Seção 2.

In [ ]:
for semente in (6, 7, 8, 9):
    rng_s = np.random.default_rng(semente)
    x_s, y_s = amostra(N_TR, rng_s)
    media_s = np.array([
        -skm.cross_val_score(modelo_poly(g), x_s.reshape(-1, 1), y_s, cv=dobras,
                             scoring="neg_mean_squared_error").mean()
        for g in graus])
    marca = "  <- a amostra deste notebook" if semente == 6 else ""
    print(f"semente {semente}: grau escolhido {graus[np.argmin(media_s)]:2d}   "
          f"EQM minimo {media_s.min():.4f}{marca}")

Quatro amostras da mesma população, três respostas diferentes: graus **4, 5, 5 e 6**.

Compare as duas grandezas. Dentro de uma amostra, a distância entre o grau 4 e o
grau 5 é de $0{,}003$ em EQM. Entre amostras, o grau escolhido anda dois degraus
inteiros. **A diferença entre modelos vizinhos não sobrevive à troca de amostra.**

É esse o argumento a favor da regra de 1-EP. Quando o empate está dentro do ruído,
desempatar por parcimônia não é mais arbitrário que desempatar pelo mínimo — e é
mais estável, porque o modelo mais simples da faixa de empate muda menos de amostra
para amostra do que o argmin.

---
## 8. O protocolo: escolher no treino, medir no teste

Até aqui a validação cruzada rodou na amostra inteira, e isso foi deliberado: as
Seções 5 a 7 estudam a **régua**, não um modelo. Comparamos a CV com o atalho da
alavancagem, com o laço feito à mão, com o risco verdadeiro — e a verdade veio da
simulação, não de um conjunto de teste. Enquanto o objeto de estudo é o estimador, e
a resposta está disponível por outro caminho, usar os 50 pontos é o certo.

Agora trocamos de papel. Você é o analista: não tem $r(x)$, não tem $\sigma^2$, e
precisa entregar duas coisas — **um modelo** e **uma estimativa honesta do risco
dele**. A Seção 7 deixou essa dívida em aberto. Ela escolheu o grau 4 pela CV, mas o
EQM daquele mínimo não serve para reportar: é o menor de dez estimativas ruidosas, e
o mínimo de várias variáveis aleatórias é enviesado para baixo. É o mesmo pecado do
erro de treino, um andar acima.

O protocolo da aula teórica, na versão de três conjuntos do [AME], é:

> **treino** ajusta &nbsp;·&nbsp; **validação** escolhe o hiperparâmetro
> &nbsp;·&nbsp; **teste**, tocado uma única vez no fim, estima o risco do vencedor.

Com validação cruzada os dois primeiros papéis se fundem — a CV *dentro do treino*
ajusta e escolhe, girando as dobras —, e o teste fica de fora do começo ao fim. O
slide da aula escreve o procedimento em quatro passos:

> 1. **separe** o conjunto de teste (uns 30%) e guarde-o;
> 2. **escolha** um ou mais bons candidatos por validação cruzada *dentro do treino*;
> 3. **reajuste** o(s) candidato(s) em **todo** o conjunto de treinamento;
> 4. **meça** no conjunto de teste, e fique com o melhor.

O passo 3 é o que se esquece, e é ele que faz a CV valer a pena: as dobras serviram
para *escolher*, não para entregar o modelo — o que se entrega treina com as 35
observações do treino, não com 28. Vamos rodar os quatro na mesma amostra de 50
pontos e, como a população é nossa, conferir contra a verdade.

In [ ]:
# passo 1: o teste sai primeiro, e so reaparece no passo 4
X_tr, X_te, y_tr, y_te = skm.train_test_split(X, y, test_size=0.3, random_state=0)

# passo 2: a CV roda DENTRO do treino, e produz os candidatos
busca = skm.GridSearchCV(
    modelo_poly(1),                       # o grau vem da grade
    {"poly__degree": np.arange(1, 11)},
    cv=skm.KFold(5, shuffle=True, random_state=0),
    scoring="neg_mean_squared_error",
)
busca.fit(X_tr, y_tr)                     # so o treino entra aqui

media_i = -busca.cv_results_["mean_test_score"]
ep_i = busca.cv_results_["std_test_score"] / np.sqrt(5)
i_min = int(np.argmin(media_i))
i_1ep = int(np.argmax(media_i <= media_i[i_min] + ep_i[i_min]))
candidatos = sorted({int(graus[i_min]), int(graus[i_1ep])})

print(f"treino: {len(y_tr)} pts   teste: {len(y_te)} pts")
print(f"passo 2: minimo da CV no grau {graus[i_min]}, regra de 1-EP no grau {graus[i_1ep]}")
print(f"         candidatos: {candidatos}")
print(f"(a) minimo da CV no treino: {media_i[i_min]:.4f}   <- nao serve para reportar")

# passos 3 e 4: reajusta cada candidato em TODO o treino, mede no teste
grade_r = np.linspace(A, B, 400)
print("\npasso 3+4: reajustado em todo o treino, medido no teste")
for g in candidatos:
    m = modelo_poly(g).fit(X_tr, y_tr)
    risco_g = np.mean((m.predict(grade_r.reshape(-1, 1)) - r(grade_r)) ** 2) + SIGMA ** 2
    print(f"  grau {g}: (b) EQM no teste {np.mean((y_te - m.predict(X_te)) ** 2):.4f}"
          f"   |   (c) risco verdadeiro {risco_g:.4f}   <- so a simulacao entrega")

Aqui o passo 4 não teve nada a decidir: o mínimo da CV e a regra de 1-EP apontaram
o **mesmo** grau, e a lista de candidatos tem um elemento só. Não é sempre assim —
nas 200 amostras a seguir eles diferem em 80 delas —, mas é o que aconteceu nesta.

E repare no que aconteceu com os números. O mínimo da CV ficou **acima** da verdade e
o teste ficou **abaixo**: o oposto do que a teoria prevê para (a). Não é
contra-exemplo, é ruído — com 15 pontos de teste e dez modelos na disputa, uma
amostra não decide nada. Repetimos 200 vezes.

Junto vai o **atalho** que a Seção 7 usaria se fosse ingênua: rodar a CV nos 50
pontos e reportar o próprio mínimo, sem nunca separar teste. É o procedimento que
queremos ver falhar.

In [ ]:
def risco_de(m):
    """Risco verdadeiro de um modelo ja ajustado, pelo truque da Secao 7."""
    return np.mean((m.predict(grade_r.reshape(-1, 1)) - r(grade_r)) ** 2) + SIGMA ** 2


def protocolo(Xb, yb, semente):
    """Os quatro passos do slide, e devolve o que cada variante entregaria."""
    Xa, Xt, ya, yt = skm.train_test_split(Xb, yb, test_size=0.3, random_state=semente)
    b = skm.GridSearchCV(modelo_poly(1), {"poly__degree": graus},
                         cv=skm.KFold(5, shuffle=True, random_state=semente),
                         scoring="neg_mean_squared_error").fit(Xa, ya)
    md = -b.cv_results_["mean_test_score"]
    e = b.cv_results_["std_test_score"] / np.sqrt(5)
    im = int(np.argmin(md))
    i1 = int(np.argmax(md <= md[im] + e[im]))
    cands = sorted({int(graus[im]), int(graus[i1])})

    ms = {g: modelo_poly(g).fit(Xa, ya) for g in cands}                 # passo 3
    eqm = {g: np.mean((yt - ms[g].predict(Xt)) ** 2) for g in cands}    # passo 4
    g_min, g_venc = int(graus[im]), min(eqm, key=eqm.get)
    return {"cv": md[im], "n_cand": len(cands),
            "te": eqm[g_min], "verdade": risco_de(ms[g_min]),
            "te_esc": eqm[g_venc], "verdade_esc": risco_de(ms[g_venc])}


def atalho_sem_teste(Xb, yb, semente):
    """CV nos 50 pontos, reportando o proprio minimo: nenhum teste e separado."""
    b = skm.GridSearchCV(modelo_poly(1), {"poly__degree": graus},
                         cv=skm.KFold(5, shuffle=True, random_state=semente),
                         scoring="neg_mean_squared_error").fit(Xb, yb)
    return -b.best_score_, risco_de(b)


rng_p = np.random.default_rng(41)
linhas, cheia = [], []
for b in range(200):
    xb, yb = amostra(N_TR, rng_p)
    Xb = xb.reshape(-1, 1)
    linhas.append(protocolo(Xb, yb, b))
    cheia.append(atalho_sem_teste(Xb, yb, b))

col = {k: np.array([d[k] for d in linhas]) for k in linhas[0]}
cv_cheia = np.array([c[0] for c in cheia])
verdade_cheia = np.array([c[1] for c in cheia])
dois = int((col["n_cand"] == 2).sum())

print(f"PROTOCOLO (treino 35 / teste 15), em 200 amostras")
print(f"  o passo 4 teve o que decidir em {dois} delas (2 candidatos)")
print(f"  (a) minimo da CV no treino: abaixo da verdade em "
      f"{100 * (col['cv'] < col['verdade']).mean():.1f}% das amostras")
print(f"  (b) EQM no teste          : abaixo da verdade em "
      f"{100 * (col['te'] < col['verdade']).mean():.1f}% das amostras")
print(f"  erro absoluto mediano     : (a) {np.median(abs(col['cv'] - col['verdade'])):.4f}"
      f"   (b) {np.median(abs(col['te'] - col['verdade'])):.4f}")
print(f"\n  e o passo 4 cobra o seu: escolhendo entre os candidatos NO TESTE, o EQM")
print(f"  de teste passa a ficar abaixo da verdade em "
      f"{100 * (col['te_esc'] < col['verdade_esc']).mean():.1f}% das amostras")

print("\nATALHO (CV nos 50 pontos, reportando o proprio minimo)")
print(f"  abaixo da verdade em {100 * (cv_cheia < verdade_cheia).mean():.1f}% das amostras")

print("\nrisco verdadeiro mediano do modelo que cada um entrega")
print(f"  protocolo (treinado em 35 pts): {np.median(col['verdade']):.4f}")
print(f"  atalho    (treinado em 50 pts): {np.median(verdade_cheia):.4f}")

In [ ]:
dif_cv = col["cv"] - col["verdade"]
dif_te = col["te"] - col["verdade"]
fora = int((abs(dif_cv) > 1).sum() + (abs(dif_te) > 1).sum())

fig, ax = subplots(figsize=(5.6, 3.0))
faixa = np.linspace(-1, 1, 41)
ax.hist(np.clip(dif_cv, -1, 1), bins=faixa, alpha=0.6, color="steelblue",
        label="(a) minimo da CV no treino")
ax.hist(np.clip(dif_te, -1, 1), bins=faixa, alpha=0.6, color="crimson",
        label="(b) EQM no teste")
ax.axvline(0, color="black", lw=1.2)
ax.set_xlabel("estimativa - risco verdadeiro")
ax.set_ylabel("frequencia")
ax.legend(fontsize=8)

print(f"mediana da diferenca: (a) {np.median(dif_cv):+.4f}   (b) {np.median(dif_te):+.4f}")
print(f"quartis de (a): [{np.percentile(dif_cv, 25):+.3f}, {np.percentile(dif_cv, 75):+.3f}]")
print(f"quartis de (b): [{np.percentile(dif_te, 25):+.3f}, {np.percentile(dif_te, 75):+.3f}]")
print(f"({fora} das 400 diferencas sairam de [-1, 1] e estao empilhadas nas bordas)")

> **A lição, em três partes.**
>
> **1. O mínimo da CV mente, e mente para baixo.** Ele fica abaixo do risco
> verdadeiro em **66,5%** das amostras — e o atalho da amostra inteira, em
> **61,0%**. A mediana da diferença é $-0{,}08$, e o quartil superior mal alcança o
> zero. Não é ruído com média zero: é viés, e tem direção.
>
> **2. O teste compra honestidade, não precisão.** O EQM no teste fica abaixo da
> verdade em **53,0%** das amostras — cara ou coroa, que é o que se espera de um
> estimador sem viés. Mas o erro absoluto mediano é **0,1443**, praticamente igual
> ao **0,1448** do mínimo da CV. Com 15 pontos de teste, o estimador honesto erra
> tanto quanto o enviesado; a diferença é que o erro dele não tem lado preferido.
>
> **3. E o teste custa.** O modelo entregue pelo protocolo treinou com 35 pontos, e
> o do atalho com 50: risco verdadeiro mediano $0{,}6241$ contra $0{,}5908$. Separar
> o teste piorou o modelo em $5{,}6\%$. São as duas limitações do *data splitting*
> que a Seção 4 já tinha medido, reaparecendo — agora no lugar onde elas doem.
>
> **4. E o passo 4 cobra um pedacinho da honestidade de volta.** Quando o teste
> também *escolhe* entre os candidatos, ele deixa de ser um espectador: o EQM de
> teste passa a ficar abaixo da verdade em **56,0%** das amostras, contra os
> $53{,}0\%$ de quando ele só mede. É pouco — são dois candidatos, não dez —, mas é
> a mesma mecânica do item 1, uma escala abaixo. Com muitos finalistas, o teste
> viraria uma segunda validação, e você precisaria de um terceiro conjunto.

Então por que o protocolo, se ele piora o modelo e não melhora a estimativa? Por
duas razões, e nenhuma das duas cabe na tabela acima.

A primeira é que os dois custos encolhem depressa com $n$, e o viés não some: o
mínimo de dez estimativas ruidosas é otimista para qualquer tamanho de amostra. Com
50 pontos o teste é caro e ruidoso; com 21 mil ele custa quase nada e a estimativa
fica apertada — é o que a próxima seção mostra, com $6\,379$ pontos de teste, onde
o erro-padrão de uma média já é pequeno.

A segunda é mais importante. **O viés do mínimo da CV você não consegue medir.**
Aqueles $66{,}5\%$ só apareceram porque existe a coluna (c), e a coluna (c) não
existe fora da simulação — em dados reais você vê (a), e nada lhe diz o quanto ele
está baixo. Já o erro do conjunto de teste é mensurável — é a incerteza de uma
média, e ela encolhe com $\sqrt{m}$. Entre um número enviesado de tamanho
desconhecido e um número ruidoso de ruído conhecido, só o segundo dá para reportar.

Duas saídas, quando os dados são poucos e separar um teste dói. A **CV aninhada** —
um laço externo estima o desempenho, um laço interno escolhe o hiperparâmetro — é a
que as notas descrevem. A outra sai de uma observação delas: quanto mais modelos
você compara na validação, mais otimista ela fica. Logo, **encurte a grade** — não
ofereça à busca opções que você já sabe que são ruins.

Uma última observação sobre o que acabou de acontecer, e que é fácil não notar: o
`GridSearchCV` **reajustou o vencedor em todo o conjunto de treino** depois de
escolher. É o `refit=True`, que é o padrão. A CV estima o risco de um
*procedimento*; o modelo que você entrega é o que esse procedimento produz com
todos os dados que ele tinha direito de ver — os 35 do treino aqui, e no caso real
da próxima seção, todo o treino.

---
## 9. No escuro: escolhendo o $\lambda$ da Ridge em dados reais

Fechamos com o `superconductivity.csv` da Aula 02: 21.263 materiais, 81
atributos, e a temperatura crítica como resposta. Aqui não há $r(x)$, não há
$\sigma^2$ e não há amostra nova — a linha (c) da seção anterior deixa de existir,
e sobram (a) e (b).

O protocolo é o mesmo da Seção 8, e agora no regime em que ele é barato: 21 mil
observações dão um teste de 6.379 pontos, grande o bastante para a estimativa
honesta ser também precisa.

In [ ]:
import os

_nome = "superconductivity.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

df = pd.read_csv(_caminho)
Xs = df.drop(columns="critical_temp").values
ys = df["critical_temp"].values
print("dimensoes:", Xs.shape)

Primeiro separamos um conjunto de teste e **guardamos a chave**. Ele não vai ser
tocado até a última célula. Toda a escolha de $\lambda$ acontece dentro do
treino, por validação cruzada.

In [ ]:
X_tr, X_te, y_tr, y_te = skm.train_test_split(Xs, ys, test_size=0.3, random_state=0)

modelo = Pipeline([("escala", StandardScaler()), ("ridge", skl.Ridge())])
alphas = np.logspace(-2, 4, 25)

busca_ridge = skm.GridSearchCV(
    modelo, {"ridge__alpha": alphas},
    cv=skm.KFold(5, shuffle=True, random_state=0),
    scoring="neg_mean_squared_error",
    return_train_score=True,
)
busca_ridge.fit(X_tr, y_tr)

print(f"alpha escolhido: {busca_ridge.best_params_['ridge__alpha']:.4g}")
print(f"EQM de CV no vencedor: {-busca_ridge.best_score_:.3f}")

A curva de CV com as barras de um erro-padrão, e as duas escolhas possíveis: o
mínimo e a regra de 1-EP.

In [ ]:
res = busca_ridge.cv_results_
media = -res["mean_test_score"]
ep = res["std_test_score"] / np.sqrt(5)

i_min = np.argmin(media)
limite = media[i_min] + ep[i_min]
i_1ep = np.max(np.where(media <= limite))      # o alpha MAIOR = modelo mais simples

fig, ax = subplots(figsize=(5.4, 3.2))
ax.errorbar(alphas, media, yerr=ep, fmt="o-", ms=3.5, capsize=2.5)
ax.axvline(alphas[i_min], ls=":", color="steelblue", label="minimo da CV")
ax.axvline(alphas[i_1ep], ls="--", color="darkorange", label="regra de 1-EP")
ax.set_xscale("log")
ax.set_xlabel("alpha (= lambda)"); ax.set_ylabel("EQM de validacao cruzada")
ax.legend(fontsize=8)

print(f"minimo da CV : alpha = {alphas[i_min]:.4g}")
print(f"regra de 1-EP: alpha = {alphas[i_1ep]:.4g}")

A curva é quase plana numa faixa larga de $\lambda$ — o que quer dizer que, neste
problema, a regularização não é o que decide o desempenho. Vale registrar isso:
nem toda busca de hiperparâmetro compensa o esforço, e a curva de CV é quem
avisa.

Agora sim, a única vez em que tocamos o teste.

In [ ]:
from sklearn.metrics import mean_squared_error

eqm_teste = mean_squared_error(y_te, busca_ridge.predict(X_te))
print(f"EQM de CV no treino (otimista): {-busca_ridge.best_score_:.3f}")
print(f"EQM no teste, medido uma vez  : {eqm_teste:.3f}")
print(f"variancia de y (preditor constante): {ys.var():.3f}")

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| erro de treino | §3 | desce sempre; o otimismo é sistemático e vale $\sigma^2(p+1)/n$ |
| *data splitting* | §4 | honesto, mas instável (200 divisões, faixa larga) e desperdiça dados |
| LOOCV | §5 | atalho da alavancagem confere com a força bruta até $10^{-15}$ |
| $k$ dobras | §6 | na mão e no `cross_val_score` dão o mesmo número |
| `shuffle` | §6 | sem embaralhar, num banco ordenado, a estimativa muda de ordem de grandeza |
| CV × risco | §7 | erra o nível, acerta o lugar do mínimo — que é o que a seleção precisa |
| regra de 1-EP | §7 | entre amostras o grau escolhido anda dois degraus; a parcimônia é mais estável |
| o protocolo | §8 | escolher no treino e medir no teste: o mínimo da CV é baixo em 2 de 3 amostras |
| o preço dele | §8 | o teste compra honestidade, não precisão — e o modelo entregue treina com menos dados |
| caso real | §9 | curva de CV plana: às vezes o hiperparâmetro não é o que decide |

**Leitura recomendada.** [AME] §1.4 e §1.5.1 — vale ler a Observação 1.5, que é a
conta de por que o LOOCV é aproximadamente não-viesado, e o trecho sobre
intervalos de confiança para o risco.
[ISLP] Capítulo 5: §5.1.1–5.1.3 (as três estratégias), §5.1.4 (o argumento
viés–variância por trás da escolha de $k$) e §5.2 (bootstrap). As Figuras 5.6 e
5.8 do livro são a versão deles da nossa figura da Seção 7.

**Para praticar.** `Lista teorica 03.pdf` (teórica, com gabarito) e
`Lista prática 03.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 04 troca o polinômio global por métodos que olham só a
vizinhança do ponto: o KNN. A ferramenta para escolher o $k$
dele é a que acabamos de montar aqui — e o protocolo da Seção 8 é o mesmo, em
todas as aulas que vêm depois.